# Project - Airline AI Assistant (Open Source / Ollama edition)

This is the `week2/day4.ipynb` project, but running entirely **locally on open source Llama** via Ollama - no OpenAI or Anthropic API keys, no cost, no data leaving your machine.

Ollama exposes an OpenAI-compatible endpoint at `/v1`, so we still use the `openai` python package - we just point it at localhost. The only real differences from the frontier-model version:

1. We set `base_url` to our local Ollama server and pass a dummy api_key.
2. Small models are less reliable at tool calling, so we check `message.tool_calls` directly rather than trusting `finish_reason`, and we give the model a firmer nudge in the system prompt.

**Prerequisite:** Ollama running, with a tool-capable model pulled:

```
ollama pull llama3.2
```

In [ ]:
import json
import sqlite3
import requests
from openai import OpenAI
import gradio as gr

## Initialization

Change `OLLAMA_PORT` below if your Ollama listens somewhere other than the default 11434.

In [ ]:
OLLAMA_PORT = 11434
OLLAMA_HOST = f"http://localhost:{OLLAMA_PORT}"
MODEL = "llama3.2"

# No API key needed - Ollama ignores it, but the openai client insists on one being present
openai = OpenAI(base_url=f"{OLLAMA_HOST}/v1", api_key="ollama")

In [ ]:
# Sanity check: is Ollama up, and does our model exist and support tools?

try:
    models = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5).json()["models"]
except Exception as e:
    raise SystemExit(f"Could not reach Ollama at {OLLAMA_HOST} - is it running? ({e})")

print(f"Ollama is up at {OLLAMA_HOST}. Models available:")
for m in models:
    tools_ok = "tools" in m.get("capabilities", [])
    print(f"  {m['name']:<20} tools: {'yes' if tools_ok else 'no'}")

names = [m["name"] for m in models]
if not any(n == MODEL or n.startswith(f"{MODEL}:") for n in names):
    print(f"\nWARNING: '{MODEL}' not found. Run:  ollama pull {MODEL}")

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

## Step 1: A plain chat, running on local Llama

Identical code to the OpenAI version - only the client and MODEL changed.

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

## Step 2: Tools

Open source models can call tools too - Llama 3.1 onwards were trained for it, and Ollama translates the OpenAI tool schema for us.

The catch: a 3B model is much more eager to *hallucinate* a price than to call the tool. So the system prompt below is more explicit than the frontier-model one.

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.

You do not know any ticket prices from memory. Whenever a customer asks about the price
of a ticket, you must call the get_ticket_price tool to look it up. Never guess a price.
Each call handles exactly one city, so if the customer asks about several cities,
call the tool once for each of them.
"""

In [ ]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}", flush=True)
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

In [ ]:
get_ticket_price("London")

In [ ]:
# The dictionary structure that describes our function.
# Note the descriptions are more emphatic than the frontier-model version needs -
# a 3B model will otherwise cheerfully pass "London,Paris" as a single city.

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to one destination city. Call this once per city; to price several cities, make one separate call for each.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "A single city that the customer wants to travel to, e.g. 'London'. Never a list of cities.",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function}]

## Step 3: Getting local Llama to use our tool

Two small departures from the frontier-model version, both aimed at the same problem - a 3B model is a lot sloppier than GPT-4.1:

1. **We check `message.tool_calls` instead of `finish_reason == "tool_calls"`.** Ollama does set `finish_reason` correctly for llama3.2, so either works here - but the tool calls themselves are the thing we actually care about, and checking them directly means this code survives a model or server that reports `finish_reason` less carefully.
2. **We pass `temperature=0`.** Tool calling is a formatting task, not a creative one. At default temperature llama3.2 intermittently mangles arguments or leaks raw tool JSON into its prose reply; at 0 it is markedly more consistent.

We also cap the tool loop, so a confused model can't spin forever.

In [ ]:
def handle_tool_calls(message):
    """Run every tool the model asked for, and return one tool message per call."""
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments or "{}")
            city = arguments.get("destination_city")
            result = get_ticket_price(city)
        else:
            result = f"Unknown tool: {tool_call.function.name}"
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses

In [ ]:
MAX_TOOL_ROUNDS = 5

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)

    # Loop, because the model may want another tool after seeing the first result
    for _ in range(MAX_TOOL_ROUNDS):
        if not response.choices[0].message.tool_calls:
            break
        assistant_message = response.choices[0].message
        messages.append(assistant_message)
        messages.extend(handle_tool_calls(assistant_message))
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## Step 4: Back the prices with a database

Same as the frontier notebook - swap the hardcoded dict for SQLite, so the tool reads real state.

In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)")
    conn.commit()

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?", (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"


def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting price for {city} to {price}", flush=True)
    price = float(price)  # a small model may hand us the number as a string
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?",
            (city.lower(), price, price)
        )
        conn.commit()
    return f"Ticket price to {city} is now set to ${price}"

In [ ]:
# Seed some data

for city, price in {"london": 799, "paris": 899, "tokyo": 1420, "sydney": 2999}.items():
    set_ticket_price(city, price)

print(get_ticket_price("Tokyo"))

## Step 5: Two tools, and a cleaner dispatch

Now the assistant can both read and write prices. Instead of an `if` chain, we map tool names to python functions - so adding a third tool is two lines.

In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to one destination city. Call this once per city; to price several cities, make one separate call for each.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "A single destination city name, e.g. 'London'. Never a list of cities.",
            },
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set or update the price of a return ticket to one destination city. Only use this when explicitly asked to change a price.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "A single destination city name the price applies to",
            },
            "price": {
                "type": "number",
                "description": "The new ticket price in US dollars",
            },
        },
        "required": ["city", "price"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function},
]

# Map the tool name the model uses onto the python function that implements it
tool_functions = {
    "get_ticket_price": get_ticket_price,
    "set_ticket_price": set_ticket_price,
}

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments or "{}")
        function = tool_functions.get(name)
        if function is None:
            result = f"Error: there is no tool called {name}"
        else:
            try:
                result = function(**arguments)
            except Exception as e:
                # Hand the error back to the model rather than crashing the chat -
                # small models get argument names wrong, and can often recover if told
                result = f"Error calling {name} with {arguments}: {e}"
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.

You do not know any ticket prices from memory. Use the get_ticket_price tool to look up a
price, and the set_ticket_price tool when asked to change one. Never guess or invent a price.
Each call handles exactly one city, so if the customer asks about several cities,
call the tool once for each of them.
"""


def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)

    for _ in range(MAX_TOOL_ROUNDS):
        if not response.choices[0].message.tool_calls:
            break
        assistant_message = response.choices[0].message
        messages.append(assistant_message)
        messages.extend(handle_tool_calls(assistant_message))
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

### Things to try

- *"How much is a ticket to Tokyo?"* - watch the tool print in the notebook output
- *"How much to London and Paris?"* - two tool calls in one turn
- *"Set the price to Berlin to 550"* then *"How much is Berlin?"* - write, then read back
- *"How much to Mars?"* - the tool returns no data, so the model should say it doesn't know

All four work on `llama3.2` with the settings above. Expect it to still be chattier than the
"no more than 1 sentence" instruction asks for, and to sometimes tack on an odd follow-up
question - that is the 3B model, not the plumbing.

### If tool calling misbehaves

Small models are the weak link here, not the code. In rough order of effectiveness:

1. **Use a bigger, tool-capable local model.** Set `MODEL = "gpt-oss:20b"` - or any model whose `capabilities` included `tools` in the check at the top - and re-run from the config cell down. Bigger models follow the length instruction better too.
2. **Keep `temperature=0`.** Removing it noticeably degrades argument quality.
3. **Be blunt in the tool descriptions.** The "one city per call, never a list" wording earns its keep at 3B: without it, llama3.2 passes `"London,Paris"` as a single city and then leaks raw tool JSON into its answer.

Note that of the models on this machine, `phi3` and `gemma3:270m` cannot call tools at all - they'll answer in prose and happily invent prices.